In [127]:
import random

import pandas as pd
from pandas import DataFrame
import numpy as np

from preferences import prefs
from user import user_prefs

# startPoint = {"lat": 30.317504,"lon": 59.927085}
# endPoint = {"lat": 30.327108, "lon": 59.935408}

POPULATION_SIZE = 3
GENERATIONS = 100

MIN_ROUTE_POINTS = 3
MAX_ROUTE_POINTS = 10

KILLOMETER_RADIUS = 0

random.seed(42)
np.random.seed(42)

# Особь:
# {
#     "route": [индексы точек из df],
#     "fitness": число
# }


In [128]:
df = pd.read_csv('data/places.csv')

**Алгоритмы генетики**

In [ ]:
from haversine import haversine

# Оптимизация конкретного маршрута
def optimize_route(route_ids, df, startPoint):
    points = (
        df[df["id"].isin(route_ids)]
        [["id", "lat", "lon"]]
        .copy()
    )

    remaining = points.to_dict("records")

    current = {
        "lat": startPoint["lat"],
        "lon": startPoint["lon"]
    }

    optimized_route = []

    while remaining:
        nearest = min(
            remaining,
            key=lambda p: haversine(
                (current["lat"], current["lon"]),
                (p["lat"], p["lon"])
            )
        )

        optimized_route.append(nearest["id"])

        current = nearest

        remaining.remove(nearest)

    return optimized_route
    

# Генерация случайной особи
def create_population(df: DataFrame, startPoint: dict[str, float], endPoint: dict[str, float], num: int):

    route_size = random.randint(
        MIN_ROUTE_POINTS,
        MAX_ROUTE_POINTS
    )

    buffer_km = KILLOMETER_RADIUS

    # Переводим километры в градусы
    lat_buffer = buffer_km / 111

    mean_lat = (startPoint["lat"] + endPoint["lat"]) / 2
    lon_buffer = buffer_km / (111 * np.cos(np.radians(mean_lat)))

    min_lat = min(startPoint["lat"], endPoint["lat"]) - lat_buffer
    max_lat = max(startPoint["lat"], endPoint["lat"]) + lat_buffer

    min_lon = min(startPoint["lon"], endPoint["lon"]) - lon_buffer
    max_lon = max(startPoint["lon"], endPoint["lon"]) + lon_buffer

    df_filtered = df[
        (df["lat"] >= min_lat) &
        (df["lat"] <= max_lat) &
        (df["lon"] >= min_lon) &
        (df["lon"] <= max_lon)
    ]

    population = []

    df_indexes = df_filtered["id"].tolist()

    for _ in range(num):
        route_id = random.sample(
            range(len(df_indexes)),
            route_size
        )
        route = []

        for i in route_id:
            route.append(df_indexes[i])
        
        new_route = optimize_route(route, df, startPoint)
        
        population.append({"route": new_route, "fitness": None})

    return population

# Длина маршрута
def route_distance(route: list)
    


# Интересность конкретной точки
def point_interest_score(point: dict, unique_user_prefs: dict[str, int]):
    point_tags = next(iter(point.values()))
    score = 0
    for category, tags in prefs.items():
        weight = unique_user_prefs[category]
        for tag in tags:
            if tag in point_tags and point_tags[tag]:
                score += weight

    return score

# Фитнес-функция
def fitness_function(route: list, unique_user_prefs: dict[str, int]):

    total_interest = 0

    for place_id in route:
        point = df.iloc[(df["id"] == place_id)].to_dict(orient='index')
        total_interest += point_interest_score(point, unique_user_prefs)

    distance = route_distance(route)

    fitness = total_interest * 20 - distance * 0.03

    return fitness

# Проведение оценки популяции (просчёт фитнес-функций каждой особи)
def evaluate_population(population: dict, unique_user_prefs: dict[str, int]):
    for individual in population:
        individual["fitness"] = fitness_function(
            individual["route"],
            unique_user_prefs
        )

**Запуск генетики**

In [130]:
startPoint = {"lat": 59.927085,"lon": 30.317504}
endPoint = {"lat": 59.935408, "lon": 30.327108}

population = create_population(df, startPoint, endPoint, POPULATION_SIZE)

unique_user_prefs = user_prefs

for generation in range(GENERATIONS):
    evaluate_population(population, unique_user_prefs)

military ['battlefield', 'historic']
religion ['cross', 'place_of_worship', 'church', 'wayside_cross', 'wayside_shrine']
architecture ['castle', 'city_gate', 'epigraph', 'fountain', 'gate', 'hospital', 'lean_to', 'archaeological_site', 'manor', 'milestone', 'ruins', 'suburb']
transport ['aircraft', 'anchor', 'locomotive', 'missile', 'propeller', 'railway_car', 'ship', 'technical_monument', 'train', 'vehicle', 'wreck', 'cannon']
sight ['artwork', 'attraction', 'bench', 'binoculars', 'boundary_stone', 'clock', 'exhibit', 'grave', 'information', 'monument', 'printing_press', 'shield', 'statue', 'stone', 'tank', 'viewpoint']
interactive ['amusement_arcadeaquariumdog_parkgallerylibraryminemuseumpicnic_sitepicnic_tablerailway_stationsaunasocial_facilityswimming_pooltheme_parktombtrail_riding_stationtrainingwater_parkzoogarden']
nutrition ['cafe', 'restaurant']
housing []
military ['battlefield', 'historic']
religion ['cross', 'place_of_worship', 'church', 'wayside_cross', 'wayside_shrine']
a

In [131]:
import folium

def hex_to_rgb(hex_str):
    """Преобразует HEX в кортеж RGB (0-255)"""
    hex_str = hex_str.lstrip('#')
    return tuple(int(hex_str[i:i+2], 16) for i in (0, 2, 4))

def rgb_to_hex(rgb):
    """Преобразует RGB в HEX строку"""
    return '#{:02x}{:02x}{:02x}'.format(int(rgb[0]), int(rgb[1]), int(rgb[2]))

def generate_gradient(start_hex, end_hex, steps):
    """Генерирует список цветов для градиента"""
    start_rgb = hex_to_rgb(start_hex)
    end_rgb = hex_to_rgb(end_hex)
    
    gradient_colors = []
    for i in range(steps):
        t = i / max(1, steps - 1)
        r = start_rgb[0] + (end_rgb[0] - start_rgb[0]) * t
        g = start_rgb[1] + (end_rgb[1] - start_rgb[1]) * t
        b = start_rgb[2] + (end_rgb[2] - start_rgb[2]) * t
        gradient_colors.append(rgb_to_hex((r, g, b)))
        
    return gradient_colors

path_colors = generate_gradient("#FF0000","#0000FF", len(population))

route_df = df[(df["id"].isin(population[0]["route"]))]
m = folium.Map(
    location=[
        route_df["lat"].mean(),
        route_df["lon"].mean()
    ],
    zoom_start=15
)

for i in range(len(population)):
    individual = population[i]
    
    route_df = pd.DataFrame()

    for j in individual["route"]:
        route_df = pd.concat([route_df, df[(df["id"] == j)]], ignore_index=True)

    # Точки маршрута
    folium.Marker(
        [startPoint["lat"], startPoint["lon"]]
    ).add_to(m)
    folium.Marker(
        [endPoint["lat"], endPoint["lon"]]
    ).add_to(m)

    route_line = [[startPoint["lat"], startPoint["lon"]]]
    route_line.extend(route_df[["lat", "lon"]].values.tolist())
    route_line.append([endPoint["lat"], endPoint["lon"]])

    # Линия маршрута
    folium.PolyLine(
        route_line,
        weight=4,
        color=path_colors[i]
    ).add_to(m)

m.save(r"visualisations\route.html")